# Module 3 — Storing signals and time series

**Who this is for:** you need to save, load or organise recordings, or you are starting a capstone track.

> **Self-check — skip this module if you can already:**
>
> - [ ] list the metadata that must travel with a signal (and why `fs` alone is not enough)
> - [ ] save an array together with its sampling rate and channel names, and load it back
> - [ ] say when to use `.npz`, CSV, EDF or WFDB
> - [ ] explain why a subject ID has to be stored with every epoch
>
> If you can tick every box, go to the next module. If not, work through the parts you can't tick. Before
> jumping to a section, run the module's **setup cell** (the first code cell): later sections use variables
> defined earlier, so run the cells above a section if you get a `NameError`.

**You will learn**
- what a recording is: samples + metadata + labels
- NumPy `.npz` files
- pandas for tables and timestamped series (and when not to use CSV)
- the standard biosignal formats: WFDB (ECG) and EDF/FIF via MNE (EEG/PSG)
- organising epochs, labels and subject IDs for machine learning

**Where the course uses this:** Lab 1 (acquisition audit), every capstone track's `load()` and `Recording`, and the leakage-safe split by subject.

*Run each cell with **Shift+Enter**, top to bottom. Nothing here needs data files; once the libraries are
installed (see the [overview](README.md#setup-first)), the notebook runs offline.*

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/farhad-abtahi/CM2013/blob/main/python_tutorial/03_storing_signals.ipynb)
&nbsp; Part of the **Python for biosignal processing** tutorial — see the [overview](README.md) to choose which modules you need.

In [ ]:
# Colab only: install the two biosignal-format libraries this module uses (already there locally if you
# installed requirements-real.txt). The versions match the course's pinned ones.
import sys, importlib.util
missing = [m for m in ("wfdb", "mne") if importlib.util.find_spec(m) is None]
if "google.colab" in sys.modules and missing:
    %pip install -q "wfdb==4.3.0" "mne==1.10.1" "pandas<3"
# Installed is not the same as compatible: if `import wfdb` below fails, see Module 6 (pinning versions).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2013)

## 1. A recording is more than an array

Samples on their own are meaningless. Keep these together, always:

| Item | Example | Why it matters |
|---|---|---|
| samples | `(n_channels, n_samples)` float array | the signal |
| sampling rate `fs` | 250 Hz | turns samples into seconds and hertz |
| channel names | `["Fpz-Cz", "EOG"]` | which sensor is which |
| units | µV, mV | a value of 50 is meaningless without units |
| start time, with time zone | `2026-10-01 22:14:05+02:00` | aligning with other recordings and events |
| annotations / labels | sleep stage per 30 s, beat types | the targets for learning |
| subject / record ID | `"S07"` | the **split unit** for honest validation |

A small, explicit structure keeps them together. A `dataclass` is a tidy way:

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Recording:
    signal: np.ndarray              # (n_channels, n_samples)
    fs: float                       # Hz
    channels: list                  # one name per channel
    units: list                     # one unit per channel (sensors can differ)
    subject: str
    start_time: str                 # ISO 8601 with time zone, e.g. "2026-10-01T22:14:05+02:00"
    annotations: list = field(default_factory=list)   # (onset_s, duration_s, description)

fs = 100
t = np.arange(30 * fs) / fs
eeg = 20e-6 * rng.standard_normal((2, t.size)) + 30e-6 * np.sin(2 * np.pi * 10 * t)   # volts
rec = Recording(eeg, fs, ["Fpz-Cz", "Pz-Oz"], ["V", "V"], "S01", "2026-10-01T22:14:05+02:00",
                annotations=[(12.0, 0.4, "electrode pop")])
print(rec.signal.shape, rec.fs, rec.channels, rec.units, rec.subject, rec.start_time, rec.annotations)

## 2. NumPy `.npz`: the simplest way to save arrays with metadata

`np.savez_compressed` stores several named arrays in one file. Small metadata can go in as arrays too.
Annotations go in as three aligned arrays (onsets, durations, descriptions). Then **check the round trip**:
load it back and assert that everything survived.

In [ ]:
import tempfile, os
folder = tempfile.mkdtemp()

def save_recording(rec, path):
    np.savez_compressed(path, signal=rec.signal, fs=rec.fs, channels=np.array(rec.channels),
                        units=np.array(rec.units), subject=rec.subject, start_time=rec.start_time,
                        # one array per annotation field, so an empty list just gives empty arrays
                        ann_onset_s=np.array([a[0] for a in rec.annotations], dtype=float),
                        ann_duration_s=np.array([a[1] for a in rec.annotations], dtype=float),
                        ann_description=np.array([a[2] for a in rec.annotations], dtype=str))

def load_recording(path):
    with np.load(path) as d:                # 'with' closes the file again when the block ends
        return Recording(d["signal"], float(d["fs"]), d["channels"].tolist(), d["units"].tolist(),
                         str(d["subject"]), str(d["start_time"]),
                         list(zip(d["ann_onset_s"].tolist(), d["ann_duration_s"].tolist(),
                                  d["ann_description"].tolist())))

def same_recording(a, b):
    return (np.array_equal(a.signal, b.signal) and a.fs == b.fs and a.channels == b.channels
            and a.units == b.units and a.subject == b.subject and a.start_time == b.start_time
            and a.annotations == b.annotations)

path = os.path.join(folder, "S01_eeg.npz")
save_recording(rec, path)
loaded = load_recording(path)
assert same_recording(loaded, rec)
print("round trip OK:", loaded.subject, loaded.start_time, loaded.annotations)

# the same round trip must work with no annotations and with several
for anns in ([], [(12.0, 0.4, "electrode pop"), (20.5, 3.0, "movement")]):
    r2 = Recording(rec.signal, rec.fs, rec.channels, rec.units, rec.subject, rec.start_time, anns)
    save_recording(r2, path)
    assert same_recording(load_recording(path), r2)
print("round trip OK with 0 and 2 annotations too")

**Good for:** your own intermediate results (filtered signals, feature matrices). **Not** a format to
share with other labs — use the standard formats below for that.

## 3. pandas: tables and timestamped series

`pandas` is the standard library for **tables** (one row per epoch or beat, one column per feature) and for
time series with **real timestamps**. It is installed in Colab; locally run `pip install "pandas<3"` if you
need it (see Module 6 for why the version matters).

In [ ]:
import pandas as pd

# A feature table: one row per 30-s epoch, with the subject and the label
features = pd.DataFrame({
    "subject": ["S01"] * 4 + ["S02"] * 4,
    "epoch":   list(range(4)) * 2,
    "alpha_power": rng.gamma(2.0, 1.0, 8).round(3),
    "delta_power": rng.gamma(3.0, 1.0, 8).round(3),
    "stage":   ["W", "N1", "N2", "N2", "W", "N2", "N3", "REM"],
})
print(features)
print(features.groupby("subject")["alpha_power"].mean())

In [ ]:
# A timestamped series: heart rate once per second, then resampled to 1-minute means
start = pd.Timestamp("2026-10-01 22:14:05")
hr = pd.Series(70 + rng.normal(0, 2, 600), index=pd.date_range(start, periods=600, freq="1s"), name="hr_bpm")
print(hr.head(3))
print(hr.resample("1min").mean().round(1).head(3))

Note the first 1-minute value is labelled **22:14:00**, although the data start at 22:14:05: pandas aligns
the bins to the clock, so the first and last bins are **partial**. Check how many samples each bin has
(`.resample("1min").count()`), decide what to do with partial bins and gaps, and give timestamps a time zone
(`pd.Timestamp(..., tz="Europe/Stockholm")`) when recordings from different places or devices meet.

**CSV** is fine for small tables like the feature table above. It is a poor format for raw signals: it is
large, slow, loses the number type, and has nowhere to put `fs` or units. A 10-hour, 2-channel EEG at 100 Hz
is 7.2 million numbers.

In [ ]:
csv_path = os.path.join(folder, "features.csv")
features.to_csv(csv_path, index=False)
print(pd.read_csv(csv_path).dtypes)

## 4. Standard biosignal formats

| Format | Typical use | Python library | Course track |
|---|---|---|---|
| **WFDB** (`.hea` + `.dat`) | ECG and other PhysioNet data | `wfdb` | ECG CinC-2017 |
| **EDF / EDF+** | EEG, PSG, sleep recordings with annotations | `mne` (`mne.io.read_raw_edf`) | Sleep-EDF |
| **FIF** | MNE's own format, keeps everything | `mne` | — |
| **`.mat`** | MATLAB files (Ninapro EMG) | `scipy.io.loadmat` (files up to v7); `h5py` for v7.3 files, which are HDF5 | EMG Ninapro |

WFDB, EDF and FIF keep the samples, `fs`, channel names and units together. A `.mat` file is just a container:
what is inside, and whether `fs` and units are stored at all, depends on the dataset, so read its
documentation (the track's dataset card). The capstone track modules already wrap
them in `download()` and `load()` — you rarely call these libraries yourself, but you should recognise them.

Below, we write a small WFDB record and read it back, entirely offline:

In [ ]:
import wfdb

fs_ecg = 250
t_ecg = np.arange(5 * fs_ecg) / fs_ecg
ecg = np.c_[np.sin(2 * np.pi * 1.2 * t_ecg), 0.5 * np.cos(2 * np.pi * 1.2 * t_ecg)]   # (n_samples, n_channels)!

wfdb.wrsamp("demo", fs=fs_ecg, units=["mV", "mV"], sig_name=["ECG I", "ECG II"],
            p_signal=ecg, fmt=["16", "16"], write_dir=folder)
r = wfdb.rdrecord(os.path.join(folder, "demo"))
print(r.fs, r.sig_name, r.units, r.p_signal.shape)
print("max round-trip error:", np.abs(r.p_signal - ecg).max())

Two things to notice:

1. **WFDB stores samples × channels**, while MNE and the course use **channels × samples**. Transpose with
   `.T` when you move between them, and check `.shape` every time.
2. The round-trip error is not zero: WFDB saved the signal as **16-bit integers** (like a real ADC) and scaled
   it back. That is quantisation (Chapter 2).

In [ ]:
import mne

info = mne.create_info(ch_names=rec.channels, sfreq=rec.fs, ch_types="eeg")
raw = mne.io.RawArray(rec.signal, info, verbose=False)      # MNE expects volts, channels x samples
print(raw)
raw.save(os.path.join(folder, "S01_raw.fif"), overwrite=True, verbose=False)
raw2 = mne.io.read_raw_fif(os.path.join(folder, "S01_raw.fif"), preload=True, verbose=False)
print(raw2.info["sfreq"], raw2.ch_names, raw2.get_data().shape)

# Reading a real EDF file looks like this (not run here: it needs a downloaded file):
# raw = mne.io.read_raw_edf("SC4001E0-PSG.edf", preload=True)

## 5. Epochs, labels and subject IDs for machine learning

Machine learning needs three aligned arrays: features `X` (one row per epoch), labels `y`, and **`groups`**
(the subject or record each row came from). `groups` is what lets you split by subject so a model is never
tested on a person it trained on (Chapter 12, and the `split unit` in every track card).

In [ ]:
epoch_s = 5
w = epoch_s * fs
X_rows, y, groups = [], [], []
for subj in ["S01", "S02", "S03"]:
    sig = 20e-6 * rng.standard_normal(60 * fs)                  # 60 s per subject
    for k in range(sig.size // w):
        e = sig[k * w:(k + 1) * w]
        X_rows.append([e.std(), np.abs(e).max()])
        y.append(k % 2)
        groups.append(subj)
X = np.array(X_rows); y = np.array(y); groups = np.array(groups)
print(X.shape, y.shape, groups.shape, np.unique(groups))

## Try it yourself

Save `X`, `y` and `groups` into one `.npz` file, load them back into `X2`, `y2` and `groups2`, and check
with `assert` that each loaded array is **identical** to the original (same values, same order), not just
the same length.

In [ ]:
# TODO

In [ ]:
# Check: keep this cell separate, so it survives when you paste in a solution.
try:
    ok = np.array_equal(X2, X) and np.array_equal(y2, y) and np.array_equal(groups2, groups)
    print("correct" if ok else "not yet: the loaded arrays differ from the originals")
except NameError:
    print("Not done yet: create X2, y2 and groups2 by loading your file.")

**Solution:** try it yourself first, then compare with the [solutions notebook](solutions.ipynb) (Module 3). Yours may differ and still be right.

**Next:** Module 4 — what to do with the signal once it is loaded.